In [ ]:
from pathlib import Path
import warnings

import pandas as pd
import numpy as np  
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from IPython.display import display, Markdown

import sim_ranking as sr
import ml_tools as mlt

In [ ]:
wdata = "/Users/claudy/dev/work/data"
results_dir = "/Users/claudy/dev/work/data/sim_ranking/results/gnn/0326_1434_cv_v4p3NZGMDB_v2p7_6e8s"

In [ ]:
wdata = Path(wdata)
results_dir = Path(results_dir) 
warnings.simplefilter(action='ignore', category=FutureWarning)

**Result Directory:** `{python} str(results_dir)`

In [ ]:
# Load observed data 
run_config = sr.ml.gnn_gm.RunConfig.from_yaml(results_dir / "run_config.yaml")
nzgmdb_ffp = wdata / run_config.rel_obs_data_ffp
obs_data = sr.data.load_obs_nzgmdb(nzgmdb_ffp)

In [ ]:
# Load GNN results
val_results = pd.read_parquet(results_dir / "val_results.parquet")

In [ ]:
# Compute residuals
val_res_df = sr.ml.gnn_gm.get_residuals(val_results, ims=run_config.ims)
assert val_res_df.index.equals(val_results.index)
val_res_df["cv_iter"] = val_results["cv_iter"]

In [ ]:
# Distance matrix
dist_matrix = sr.utils.calculate_distance_matrix(obs_data.sites, obs_data.site_df)

In [ ]:
# Compute residuals per site
gnn_mean_site_res_df = val_res_df.groupby("site_int")[run_config.ims].mean().sort_index()
gnn_std_site_res_df = val_res_df.groupby("site_int")[run_config.ims].std().sort_index()
gnn_site_scenario_count = val_res_df.site_int.value_counts().sort_index()

# Drop any with less than 10 scenarios
mask = gnn_site_scenario_count >= 10
gnn_mean_site_res_df = gnn_mean_site_res_df.loc[mask]
gnn_std_site_res_df = gnn_std_site_res_df.loc[mask]
gnn_site_scenario_count = gnn_site_scenario_count.loc[mask]

In [ ]:
cur_im = "pSA_0.1"

ind_keys = gnn_mean_site_res_df[cur_im].sort_values().index

fig, ax = plt.subplots(1, 1, figsize=(16, 6))

ax.bar(np.arange(len(gnn_mean_site_res_df)), gnn_mean_site_res_df.loc[ind_keys, cur_im], yerr=gnn_std_site_res_df.loc[ind_keys, cur_im], alpha=0.5, color="blue", label="GNN")
ax.set_xlim(-1, len(gnn_mean_site_res_df))

fig.tight_layout()